## Imports

In [ ]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
from google.colab import userdata

## Loading the API key

In [ ]:
load_dotenv(override=True)
google_api_key = userdata.get('GOOGLE_API_KEY2') # Using userdata for Colab secrets
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

### Connect to OpenAI client library

In [ ]:
# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

# Initialize specific clients first
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url) if anthropic_api_key else None
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url) if google_api_key else None
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url) if deepseek_api_key else None
groq = OpenAI(api_key=groq_api_key, base_url=groq_url) if groq_api_key else None
grok = OpenAI(api_key=grok_api_key, base_url=grok_url) if grok_api_key else None
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key) if openrouter_api_key else None
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

# Now, initialize the 'openai' client variable conditionally
# Prioritize actual OpenAI key, then fallback to Gemini if only Google key is available
if openai_api_key:
    openai = OpenAI(api_key=openai_api_key)
    print("OpenAI client initialized with OpenAI API key.")
elif google_api_key:
    # Use the Gemini client as a fallback for 'openai' if google_api_key is available
    # This allows using 'openai' for Gemini-backed calls.
    openai = gemini
    print("OpenAI API Key not set. 'openai' client variable now refers to the Gemini client (using Google API key).")
else:
    openai = None # Explicitly set to None if no suitable API key is found
    print("Neither OpenAI nor Google API Key set. 'openai' client not initialized.")

### Prompt

In [ ]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

In [ ]:
if google_api_key:
    # Use the 'gemini_url' variable which was set up for OpenAI compatibility.
    correct_gemini_client = OpenAI(api_key=google_api_key, base_url=gemini_url)
    # Removed 'models/' prefix from the model name as it's often not required for chat completions
    response = correct_gemini_client.chat.completions.create(model="gemini-2.5-flash", messages=tell_a_joke)
    display(Markdown(response.choices[0].message.content))
else:
    display(Markdown("Google API key not found. Cannot call Gemini API."))

In [ ]:
# if google_api_key:
#     # Use the 'gemini_url' variable which was set up for OpenAI compatibility.
#     correct_gemini_client = OpenAI(api_key=google_api_key, base_url=gemini_url)
#     print("Listing available models from Gemini API via OpenAI client:")
#     try:
#         models = correct_gemini_client.models.list()
#         for model in models.data:
#             print(f"- {model.id}")
#     except Exception as e:
#         print(f"Error listing models: {e}")
# else:
#     print("Google API key not found. Cannot list models.")

### Training vs Inference time scaling

In [ ]:
easy_puzzle = [
    {"role": "user", "content":
        "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."},
]

In [ ]:
response = openai.chat.completions.create(model="gemini-2.5-flash", messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

In [ ]:
response = openai.chat.completions.create(model="gemini-2.5-flash", messages=easy_puzzle, reasoning_effort="low")
display(Markdown(response.choices[0].message.content))

### Gemini Library

In [ ]:
from google import genai

client = genai.Client(api_key=google_api_key)

response = client.models.generate_content(
    model="gemini-2.5-flash", contents="Describe the color Blue to someone who's never been able to see in 1 sentence"
)
print(response.text)

# Gradio


In [ ]:
import gradio as gr

In [ ]:
system_message = "You are a helpful assistant"

def message_gpt(prompt):
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    response = openai.chat.completions.create(model="gemini-2.5-flash", messages=messages)
    return response.choices[0].message.content

In [ ]:
message_gpt("What is today's date?")

In [ ]:
def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()
shout("hello")

In [ ]:
# gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch()

In [ ]:
# gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

In [ ]:
system_message = "You are a helpful assistant"

In [ ]:
def chat(message, history):
    return "bananas"

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

### Day 4

In [ ]:
"We got to see many tools calling but due to quota limit the practise for it was not done "

### Day 5

In [ ]:
import os
import json
from dotenv import load_dotenv

import gradio as gr
import sqlite3

import requests

from IPython.display import Markdown, display
from google.colab import userdata

In [ ]:
load_dotenv(override=True)

google_api_key = userdata.get('GOOGLE_API_KEY2')
if google_api_key:
    print(f" API Key exists and begins {google_api_key[:8]}")
else:
    print(" API Key not set")

MODEL = "gemini-2.5-flash"
#openai = OpenAI()

DB = "prices.db"

In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
import sqlite3

DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    # Create the prices table if it doesn't exist
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS prices (
            city TEXT PRIMARY KEY,
            price INTEGER
        )
    ''')

    # Insert some sample data
    sample_prices = [
        ('paris', 500),
        ('london', 400),
        ('new york', 700),
        ('tokyo', 900),
        ('sydney', 1200)
    ]

    # Use executemany for efficiency and to prevent SQL injection
    cursor.executemany('INSERT OR IGNORE INTO prices (city, price) VALUES (?, ?)', sample_prices)

    conn.commit()

print(f"Database '{DB}' created/updated with 'prices' table and sample data.")

In [ ]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [ ]:
get_ticket_price("Paris")

In [ ]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    return response.choices[0].message.content

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
import base64
from io import BytesIO
from PIL import Image

In [ ]:
def artist(city):
    image_response = openai.images.generate(
            model="dall-e-3",
            prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [ ]:
image = artist("New York City")
display(image)